# Proxy Wall Finder — DAP depth generation (Colab)

Runs the [DAP](https://github.com/Insta360-Research-Team/DAP) panoramic depth model on a T4 GPU and produces the `.npz` depth files the web app reads. Our laptop GPU (4 GB) is too small for DAP; this notebook is the documented cloud path (docs/MODEL-DAP.md §4).

**Runtime → Change runtime type → T4 GPU** before running.

Cells in order: GPU check → setup → weights → **upload panos** → generate → preview → **download zip**.
Free tier is enough: 10 panos run in a couple of minutes. Panos are uploaded to Google's cloud for this run — fine for internal/dev, use a local GPU box for sensitive client sites.

In [ ]:
# 1 · GPU check
import torch
assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({vram:.0f} GB) — DAP needs >= 8 GB, we're fine.")

In [ ]:
# 3 · Download DAP weights (model.pth, 1.46 GB — 1–3 min on Colab)
import os
os.makedirs("/content/DAP/weights", exist_ok=True)
!wget -q --show-progress -O /content/DAP/weights/model.pth https://huggingface.co/Insta360-Research/DAP-weights/resolve/main/model.pth
size = os.path.getsize("/content/DAP/weights/model.pth") / 1e9
assert size > 1.0, f"Download looks truncated ({size:.2f} GB) — re-run this cell"
print(f"weights OK: {size:.2f} GB")

In [ ]:
# 3 · Download DAP weights (model.pth, 1.46 GB — 1–3 min on Colab)
import os
os.makedirs("/content/DAP/weights", exist_ok=True)
!wget -q --show-progress -O /content/DAP/weights/model.pth \
  https://huggingface.co/Insta360-Research/DAP-weights/resolve/main/model.pth
size = os.path.getsize("/content/DAP/weights/model.pth") / 1e9
assert size > 1.0, f"Download looks truncated ({size:.2f} GB) — re-run this cell"
print(f"weights OK: {size:.2f} GB")

In [ ]:
# 4 · Panoramas in — pick ONE option
#
# Option A (default): upload equirect JPG/PNG/WEBP from this computer
import os
from google.colab import files
os.makedirs("/content/panos", exist_ok=True)
uploaded = files.upload()  # select all your pano .jpg files at once
for name, data in uploaded.items():
    with open(f"/content/panos/{name}", "wb") as f:
        f.write(data)
print(f"{len(uploaded)} pano(s) in /content/panos")
#
# Option B: Google Drive (better for many panos / re-runs) — comment out A,
# uncomment below and put your panos in Drive at /MyDrive/panos
# from google.colab import drive
# drive.mount("/content/drive")
# PANOS = "/content/drive/MyDrive/panos"  # then pass this as --panos in cell 5

In [ ]:
# 5 · Generate depth — runs the repo's sidecar (scripts/generate-depth.py)
!python /content/proxy-wall-finder/scripts/generate-depth.py \
  --dap-root /content/DAP \
  --panos    /content/panos \
  --out      /content/depth

In [ ]:
# 6 · Eyeball check — depth previews + provenance
import glob, json, math
import matplotlib.pyplot as plt
import numpy as np

previews = sorted(glob.glob("/content/depth/preview/*.png"))
assert previews, "No previews — check cell 5 output"
cols = 2
rows = math.ceil(len(previews) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 4.2 * rows))
flat = list(np.atleast_1d(axes).flat)
for ax, path in zip(flat, previews):
    scene = path.split("/")[-1][:-4]
    ax.imshow(plt.imread(path))
    ax.set_title(scene)
    ax.axis("off")
for ax in flat[len(previews):]:
    ax.axis("off")
plt.show()
for path in sorted(glob.glob("/content/depth/*.json")):
    with open(path) as f:
        meta = json.load(f)
    w, h = meta["depth_resolution"][1], meta["depth_resolution"][0]
    print(f"{meta['scene']}: {w}x{h}, {meta['seconds']}s, device {meta['device']}")

In [ ]:
# 7 · Bring it home — zip and download (lands in your Downloads folder;
# unzip into proxy-wall-finder/depth/ for the web app)
from google.colab import files
!cd /content && zip -qr depth-output.zip depth
files.download("/content/depth-output.zip")